In [15]:
import pandas as pd
import numpy as np
import argparse
import time
import pickle as pkl
import os
import glob
from sklearn.impute import SimpleImputer

parser = argparse.ArgumentParser(description='Compute temporal variation features from sparse dataset for google version')


parser.add_argument('--inputdir',type=str,
                    help='Provide directory where features are located')
parser.add_argument('--outputdir',type=str,
                    help='Where should the output be stored?')
parser.add_argument('--reddy90', type=str,
                    help='Path to Reddy data')
parser.add_argument('--cordeiro90', type=str,
                    help='Path to Cordeiro 90 data')
parser.add_argument('--cordeiro100', type=str,
                    help='Path to Cordeiro 100 data')
parser.add_argument('--tag', action='store_true',
                    help='Should the POS tag be kept?')
parser.add_argument('--ppmi', action='store_true',
                    help='Should co-occurence matrix be converted to PPMI values')
parser.add_argument('--temporal',  type=int,
                    help='Value to bin the temporal information: 10 (binning to decades), 20 (binning each 20 years), 50 (binning each 50 years) or 100')
parser.add_argument('--cutoff', type=int, default=0,
                    help='Cut-off frequency for each compound per time period : none (0), 10, 50, 100, 500 and 1000')


args = parser.parse_args('--inputdir ../Compounding_datasets/ --outputdir /mount/studenten/DH_Master/compounding/google/ --temporal 100 --reddy90 ./data/reddy_90.txt --cordeiro90 ./data/cordeiro_90.txt --cordeiro100 ./data/cordeiro_100.txt'.split())

In [2]:
def testset_tagger(df):

    #### NOUN NOUN
    
    copy_df_1=df.copy()
    copy_df_1.modifier=copy_df_1.modifier+'_NOUN'
    copy_df_1['head']=copy_df_1['head']+'_NOUN'
    
    ### ADJ/NOUN NOUN
    
    copy_df_5=df.copy()
    
    copy_df_5.loc[copy_df_5.is_adj==True,"modifier"]+="_ADJ"
    copy_df_5.loc[copy_df_5.is_adj==False,"modifier"]+="_NOUN"
    copy_df_5['head']=copy_df_5['head']+'_NOUN'   
    
    complete_df=pd.concat([copy_df_1,copy_df_5],ignore_index=True)
    return complete_df    

def compound_tag_remover(compounds):
    
    print('Removing tags for compound dataset')
    compounds['head']=compounds['head'].str.replace('_NOUN|_PROPN','',regex=True)
    compounds.modifier=compounds.modifier.str.replace('_NOUN|_PROPN|_ADJ','',regex=True)
    compounds=compounds.groupby(['modifier','head','context'])['count'].sum().to_frame().reset_index()

    return compounds


def constituent_tag_remover(constituents,ctype='word'):
    
    print(f'Removing tags for {ctype} dataset')
    constituents[ctype]=constituents[ctype].str.replace('_NOUN|_PROPN|_ADJ','',regex=True)
    constituents=constituents.groupby([ctype,'context'])['count'].sum().to_frame().reset_index()

    return constituents


def process_cutoff_compound(df):
    df=df.loc[df.groupby(['modifier','head','time'])['count'].transform('sum').gt(args.cutoff)]
    
    return df


def process_cutoff_constituent(df,ctype='word'):
    df=df.loc[df.groupby([ctype,'time'])['count'].transform('sum').gt(args.cutoff)]
    
    return df



def ppmi(ppmi_df):
    
    ppmi_cols=ppmi_df.columns.tolist()
    ppmi_cols=['XY' if 'count' in x else x for x in ppmi_cols]
    ppmi_df.columns=ppmi_cols

    ppmi_time_counts=ppmi_df.groupby('time')['XY'].sum().to_frame()
    ppmi_time_counts.columns=['N']


    Y_star=ppmi_df.groupby(['context','time'])['XY'].sum().to_frame()
    Y_star.columns=['Y']

    ppmi_df=pd.merge(ppmi_df,Y_star.reset_index(),on=['context','time'])
    
    X_cols=[x for x in ppmi_cols if x not in ['context','XY'] ]


    X_star=ppmi_df.groupby(X_cols)['XY'].sum().to_frame()
    X_star.columns=['X']

    ppmi_df=pd.merge(ppmi_df,X_star.reset_index(),on=X_cols)
    ppmi_df=pd.merge(ppmi_df,ppmi_time_counts.reset_index(),on=['time'])
    ppmi_df['count']=np.log2((ppmi_df['XY']*ppmi_df['N'])/(ppmi_df['X']*ppmi_df['Y']))
    ppmi_df=ppmi_df.loc[ppmi_df['count']>=0]
    ppmi_df.drop(['XY','X','Y','N'],axis=1,inplace=True)
    return ppmi_df

def process_decades_compound(dec_list,modifier_list,head_list,input_dir,ctype='compound'):

    if os.path.exists(f"{input_dir}/{ctype}s/{args.temporal}_{dec_list[0]}_{tag_str}.pkl.bz2"):
        print(f'Reading file {ctype}')
        complete_df=pd.read_pickle(f"{input_dir}/{ctype}s/{args.temporal}_{dec_list[0]}_{tag_str}.pkl.bz2")
    elif os.path.exists(f"{input_dir}/{ctype}s/10_{dec_list[0]}_{tag_str}.pkl.bz2") and args.temporal!=10000:
        print(f'Reading decades file {ctype}s/10_{dec_list[0]}_{tag_str}.pkl.bz2')
        complete_df=pd.read_pickle(f"{input_dir}/{ctype}s/10_{dec_list[0]}_{tag_str}.pkl.bz2")
        
        print(f'Reducing to {args.temporal}')
        complete_df['time']=complete_df['time']-complete_df['time']%args.temporal

        complete_df=complete_df.groupby(['modifier','head','time','context'])['count'].sum().to_frame().reset_index()
        
        print("Saving file")
        complete_df.to_pickle(f"{input_dir}/{ctype}s/{args.temporal}_{dec_list[0]}_{tag_str}.pkl.bz2")

    else:

        df_list=[]

        for dec in dec_list:
            print(dec)
            cur_df=pd.read_pickle(f'{input_dir}/{ctype}s/{dec}.pkl.bz2')
            
            if not args.tag:
                cur_df=compound_tag_remover(cur_df)
            cur_df['time']=dec
            cur_df['time']=cur_df['time']-cur_df['time']%args.temporal
            df_list.append(cur_df)

        print('Done reading compound dataframes')
        complete_df=pd.concat(df_list,ignore_index=True)

        if args.temporal!=10:
            complete_df=complete_df.groupby(['modifier','head','time','context'])['count'].sum().to_frame().reset_index()
        
        print("Saving file")
        complete_df.to_pickle(f"{input_dir}/{ctype}s/{args.temporal}_{dec_list[0]}_{tag_str}.pkl.bz2")

    complete_df['count']=complete_df['count'].astype('float64')

    if args.cutoff==0:
        print('No cut-off applied')          
    else:
        print(f'Cut-off: {args.cutoff}')
        complete_df=process_cutoff_compound(complete_df)

    if args.ppmi:
        print('Applying PPMI')
        complete_df=ppmi(complete_df)

    print('Done processing compound dataframes')

    reduced_df=complete_df.loc[(complete_df.modifier.isin(modifier_list))&(complete_df['head'].isin(head_list))]
    
    return reduced_df


def process_decades_constituent(dec_list,constituent_list,input_dir,ctype='word'):
        
    if os.path.exists(f"{input_dir}/{ctype}s/{args.temporal}_{dec_list[0]}_{tag_str}.pkl.bz2"):
        print(f'Reading file {ctype}')
        complete_df=pd.read_pickle(f"{input_dir}/{ctype}s/{args.temporal}_{dec_list[0]}_{tag_str}.pkl.bz2")
        
    elif os.path.exists(f"{input_dir}/{ctype}s/10_{dec_list[0]}_{tag_str}.pkl.bz2") and args.temporal!=10000:
        print(f'Reading decades file {ctype}s/10_{dec_list[0]}_{tag_str}.pkl.bz2')
        complete_df=pd.read_pickle(f"{input_dir}/{ctype}s/10_{dec_list[0]}_{tag_str}.pkl.bz2")
        
        print(f'Reducing to {args.temporal}')
        complete_df['time']=complete_df['time']-complete_df['time']%args.temporal
        complete_df=complete_df.groupby([ctype,'time','context'])['count'].sum().to_frame().reset_index()
        
        print("Saving file")
        complete_df.to_pickle(f"{input_dir}/{ctype}s/{args.temporal}_{dec_list[0]}_{tag_str}.pkl.bz2")

    else:

        df_list=[]

        for dec in dec_list:
            cur_df=pd.read_pickle(f'{input_dir}/{ctype}s/{dec}.pkl.bz2')
            if not args.tag:
                cur_df=constituent_tag_remover(cur_df,ctype)
            cur_df['time']=dec
            cur_df['time']=cur_df['time']-cur_df['time']%args.temporal
            df_list.append(cur_df)

        print(f'Done reading {ctype} dataframes')
        complete_df=pd.concat(df_list,ignore_index=True)
        
        if args.temporal!=10:
            complete_df=complete_df.groupby([ctype,'time','context'])['count'].sum().to_frame().reset_index()
        
        print("Saving file")
        complete_df.to_pickle(f"{input_dir}/{ctype}s/{args.temporal}_{dec_list[0]}.pkl.bz2")

    complete_df['count']=complete_df['count'].astype('float64')

    if args.cutoff==0:
        print('No cut-off applied')          
    else:
        print(f'Cut-off: {args.cutoff}')
        complete_df=process_cutoff_constituent(complete_df,ctype=ctype)

    if args.ppmi:
        print('Applying PPMI')
        complete_df=ppmi(complete_df)

    print(f'Done processing {ctype} dataframes')
    
    reduced_df=complete_df.loc[(complete_df[ctype].isin(constituent_list))]
    
    return reduced_df

def cosine_bw_rows(df):
    
    df_orig=df.copy()
    df_shifted=df.shift().copy()
    denom_df_orig=(df_orig**2).sum(axis=1)
    denom_df_shifted=(df_shifted**2).sum(axis=1)
    denominator=np.sqrt(denom_df_orig*denom_df_shifted)
    numerator=(df_orig*df_shifted).sum(axis=1)
    if df.index.nlevels==3:
        cosine_sim_df=(numerator/denominator).reset_index(level=[0,1],drop=True)
    else:
        cosine_sim_df=(numerator/denominator).reset_index(level=[0],drop=True)        
    cosine_sim_df.dropna(inplace=True)
    cosine_sim_df=cosine_sim_df.to_frame()
    return cosine_sim_df


def temporal_features(compounds,modifiers,heads,compound_list_df):
    
    compounds_pivot=pd.pivot_table(compounds, values='count', index=['modifier','head', 'time'],
                       columns=['context'], aggfunc="sum",fill_value=0)
    modifiers_pivot=pd.pivot_table(modifiers, values='count', index=['modifier','time'],
                       columns=['context'], aggfunc="sum",fill_value=0)
    heads_pivot=pd.pivot_table(heads, values='count', index=['head','time'],
                       columns=['context'], aggfunc="sum",fill_value=0)
    
    change_compounds_df=compounds_pivot.groupby(level=[0,1]).apply(cosine_bw_rows)
    change_compounds_df.columns=['change_comp']

    change_modifiers_df=modifiers_pivot.groupby(level=[0]).apply(cosine_bw_rows)
    change_modifiers_df.columns=['change_mod']
    change_heads_df=heads_pivot.groupby(level=[0]).apply(cosine_bw_rows)
    change_heads_df.columns=['change_head']
    
    
    changed_df=pd.merge(change_compounds_df.reset_index(),compound_list_df,on=['modifier','head','time'],how='right')
    changed_df=pd.merge(changed_df,change_modifiers_df.reset_index(),on=['modifier','time'],how='right')
    
    changed_df=pd.merge(changed_df,change_heads_df.reset_index(),on=['head','time'])
    return changed_df

def merge_comp_ratings(features_df):

    features_df=pd.pivot_table(features_df, index=['modifier','head'], columns=['time'])
    features_df_columns_1=features_df.columns.get_level_values(0)
    features_df_columns_2=features_df.columns.get_level_values(1)

    cur_year=0
    new_columns=[]
    for year in features_df_columns_2:
        new_columns.append(features_df_columns_1[cur_year]+":"+str(year))
        cur_year+=1

    features_df.columns=new_columns
    cur_ratings_df_na=features_df.reset_index().merge(comp_ratings_df,on=['modifier','head'],how='right')

    imputer= SimpleImputer(strategy="median")
    df_med=pd.DataFrame(imputer.fit_transform(features_df))
    df_med.columns=features_df.columns
    df_med.index=features_df.index

    cur_ratings_df_med=df_med.reset_index().merge(comp_ratings_df,on=['modifier','head'],how='right')
    
    return cur_ratings_df_na,cur_ratings_df_med

In [5]:
reddy_df=pd.read_csv(args.reddy90,sep='\t')
reddy_df['source']='reddy'
cordeiro90_df=pd.read_csv(args.cordeiro90,sep='\t')
cordeiro90_df['source']='cordeiro90'
cordeiro100_df=pd.read_csv(args.cordeiro100,sep='\t')
cordeiro100_df['source']='cordeiro100'

comp_ratings_df=pd.concat([reddy_df,cordeiro90_df,cordeiro100_df])

if args.tag:
    comp_ratings_df=testset_tagger(comp_ratings_df)


total_dec_list=[[1820,1830,1840,1850,1860,1870,1880,1890],[1900,1910,1920,1930,1940,1950,1960,1970,1980,1990],[2000,2010]]
    
    
if args.ppmi:
    ppmi_str="PPMI"
else:
    ppmi_str="RAW"
    
if args.tag:
    tag_str='Tagged'
else:
    tag_str='UnTagged'
    

unique_modifier_list=comp_ratings_df[['modifier']].drop_duplicates()['modifier'].to_list()
unique_head_list=comp_ratings_df[['head']].drop_duplicates()['head'].to_list()
unique_constituent_list=list(set(unique_modifier_list+unique_head_list))

compounds_agnostic_list=[]
constituents_list=[]
compounds_aware_list=[]
modifiers_aware_list=[]
heads_aware_list=[]

In [6]:
for dec_list in total_dec_list:
    
    print(f'Current dec list {dec_list}')
    
    cur_compounds_agnostic=process_decades_compound(dec_list,unique_modifier_list,unique_head_list,f'{args.inputdir}',ctype="phrase")
    cur_constituents=process_decades_constituent(dec_list,unique_constituent_list,f'{args.inputdir}',ctype='word')
    
    cur_compounds_aware=process_decades_compound(dec_list,unique_modifier_list,unique_head_list,f'{args.inputdir}',ctype="compound")

    cur_modifiers_aware=process_decades_constituent(dec_list,unique_modifier_list,f'{args.inputdir}',ctype='modifier')

    cur_heads_aware=process_decades_constituent(dec_list,unique_head_list,f'{args.inputdir}',ctype='head')
    
    compounds_agnostic_list.append(cur_compounds_agnostic)
    constituents_list.append(cur_constituents)
    
    compounds_aware_list.append(cur_compounds_aware)
    modifiers_aware_list.append(cur_modifiers_aware)
    heads_aware_list.append(cur_heads_aware)
    
    
compounds_agnostic=pd.concat(compounds_agnostic_list,ignore_index=True)
constituents=pd.concat(constituents_list,ignore_index=True)

compounds_aware=pd.concat(compounds_aware_list,ignore_index=True)
modifiers_aware=pd.concat(modifiers_aware_list,ignore_index=True)
heads_aware=pd.concat(heads_aware_list,ignore_index=True)

Current dec list [1820, 1830, 1840, 1850, 1860, 1870, 1880, 1890]
Reading file phrase
No cut-off applied
Done processing compound dataframes
Reading file word
No cut-off applied
Done processing word dataframes
Reading file compound
No cut-off applied
Done processing compound dataframes
Reading file modifier
No cut-off applied
Done processing modifier dataframes
Reading file head
No cut-off applied
Done processing head dataframes
Current dec list [1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990]
Reading file phrase
No cut-off applied
Done processing compound dataframes
Reading file word
No cut-off applied
Done processing word dataframes
Reading file compound
No cut-off applied
Done processing compound dataframes
Reading file modifier
No cut-off applied
Done processing modifier dataframes
Reading file head
No cut-off applied
Done processing head dataframes
Current dec list [2000, 2010]
Reading file phrase
No cut-off applied
Done processing compound dataframes
Reading file word

In [7]:
temp_cutoff_str=str(args.temporal)+'_'+str(args.cutoff)
    
timespan_list_aware_df=pd.DataFrame(compounds_aware.time.unique())
timespan_list_aware_df.columns=['time']

compound_list_aware_df=comp_ratings_df[['modifier','head']].copy()
compound_list_aware_df=compound_list_aware_df.merge(timespan_list_aware_df,how='cross')

modifier_list_aware_df=comp_ratings_df[['modifier']].drop_duplicates().copy()
modifier_list_aware_df=modifier_list_aware_df.merge(timespan_list_aware_df,how='cross')

head_list_aware_df=comp_ratings_df[['head']].drop_duplicates().copy()
head_list_aware_df=head_list_aware_df.merge(timespan_list_aware_df,how='cross')

all_comps_aware=compounds_aware[['modifier','head','time']].copy()
all_comps_aware.drop_duplicates(inplace=True)

all_mods_aware=compounds_aware[['modifier','time']].copy()
all_mods_aware.drop_duplicates(inplace=True)

all_heads_aware=compounds_aware[['head','time']].copy()
all_heads_aware.drop_duplicates(inplace=True)

not_found_compounds_aware_df=compound_list_aware_df.merge(all_comps_aware, on=['modifier','head','time'], how='outer', suffixes=['', '_'], indicator=True)
not_found_compounds_aware_df=not_found_compounds_aware_df.loc[not_found_compounds_aware_df['_merge']=='left_only']
not_found_compounds_aware_df.drop('_merge',axis=1,inplace=True)


not_found_modifiers_aware_df=modifier_list_aware_df.merge(all_mods_aware, on=['modifier','time'], how='outer', suffixes=['', '_'], indicator=True)
not_found_modifiers_aware_df=not_found_modifiers_aware_df.loc[not_found_modifiers_aware_df['_merge']=='left_only']
not_found_modifiers_aware_df.drop('_merge',axis=1,inplace=True)

not_found_heads_aware_df=head_list_aware_df.merge(all_heads_aware, on=['head','time'], how='outer', suffixes=['', '_'], indicator=True)
not_found_heads_aware_df=not_found_heads_aware_df.loc[not_found_heads_aware_df['_merge']=='left_only']
not_found_heads_aware_df.drop('_merge',axis=1,inplace=True)



timespan_list_agnostic_df=pd.DataFrame(compounds_agnostic.time.unique())
timespan_list_agnostic_df.columns=['time']

compound_list_agnostic_df=comp_ratings_df[['modifier','head']].copy()
compound_list_agnostic_df=compound_list_agnostic_df.merge(timespan_list_agnostic_df,how='cross')

modifier_list_agnostic_df=comp_ratings_df[['modifier']].drop_duplicates().copy()
modifier_list_agnostic_df=modifier_list_agnostic_df.merge(timespan_list_agnostic_df,how='cross')

head_list_agnostic_df=comp_ratings_df[['head']].drop_duplicates().copy()
head_list_agnostic_df=head_list_agnostic_df.merge(timespan_list_agnostic_df,how='cross')

all_comps_agnostic=compounds_agnostic[['modifier','head','time']].copy()
all_comps_agnostic.drop_duplicates(inplace=True)

all_mods_agnostic=compounds_agnostic[['modifier','time']].copy()
all_mods_agnostic.drop_duplicates(inplace=True)

all_heads_agnostic=compounds_agnostic[['head','time']].copy()
all_heads_agnostic.drop_duplicates(inplace=True)

not_found_compounds_agnostic_df=compound_list_agnostic_df.merge(all_comps_agnostic, on=['modifier','head','time'], how='outer', suffixes=['', '_'], indicator=True)
not_found_compounds_agnostic_df=not_found_compounds_agnostic_df.loc[not_found_compounds_agnostic_df['_merge']=='left_only']
not_found_compounds_agnostic_df.drop('_merge',axis=1,inplace=True)

not_found_modifiers_agnostic_df=modifier_list_agnostic_df.merge(all_mods_agnostic, on=['modifier','time'], how='outer', suffixes=['', '_'], indicator=True)
not_found_modifiers_agnostic_df=not_found_modifiers_agnostic_df.loc[not_found_modifiers_agnostic_df['_merge']=='left_only']
not_found_modifiers_agnostic_df.drop('_merge',axis=1,inplace=True)

not_found_heads_agnostic_df=head_list_agnostic_df.merge(all_heads_agnostic, on=['head','time'], how='outer', suffixes=['', '_'], indicator=True)
not_found_heads_agnostic_df=not_found_heads_agnostic_df.loc[not_found_heads_agnostic_df['_merge']=='left_only']
not_found_heads_agnostic_df.drop('_merge',axis=1,inplace=True)

compounds_aware=compounds_aware.merge(comp_ratings_df[['modifier','head']],on=['modifier','head'])

compounds_agnostic=compounds_agnostic.merge(comp_ratings_df[['modifier','head']],on=['modifier','head'])

heads_agnostic=constituents.copy()
heads_agnostic_cols=heads_agnostic.columns
heads_agnostic_cols=['head' if 'word' in x else x for x in heads_agnostic_cols]
heads_agnostic.columns=heads_agnostic_cols
heads_agnostic=heads_agnostic.loc[heads_agnostic['head'].isin(unique_head_list)]


modifiers_agnostic=constituents.copy()
modifiers_agnostic_cols=modifiers_agnostic.columns
modifiers_agnostic_cols=['modifier' if 'word' in x else x for x in modifiers_agnostic_cols]
modifiers_agnostic.columns=modifiers_agnostic_cols
modifiers_agnostic=modifiers_agnostic.loc[modifiers_agnostic.modifier.isin(unique_modifier_list)]

In [8]:
print('Calculating features')

print('CompoundAware features')

change_aware_df=temporal_features(compounds_aware,modifiers_aware,heads_aware,all_comps_aware)

print('CompoundAgnostic features')

change_agnostic_df=temporal_features(compounds_agnostic,modifiers_agnostic,heads_agnostic,all_comps_agnostic)

Calculating features
CompoundAware features
CompoundAgnostic features


In [10]:
cur_ratings_aware_df_na,cur_ratings_aware_df_med=merge_comp_ratings(change_aware_df)
cur_ratings_agnostic_df_na,cur_ratings_agnostic_df_med=merge_comp_ratings(change_agnostic_df)

In [14]:
cur_ratings_aware_df_med

,modifier,head,change_comp:1900,change_comp:2000,change_head:1900,change_head:2000,change_mod:1900,change_mod:2000,avgModifier,stdevModifier,avgHead,stdevHead,compositionality,stdevHeadModifier,is_adj,compound,source,is_original
0,end,user,0.124103,0.858735,0.383222,0.776775,0.370093,0.824141,3.866667,1.117537,4.866667,0.339935,4.250000,0.871165,False,end_user,reddy,True
1,firing,line,0.687901,0.832834,0.897225,0.856429,0.854366,0.841634,1.607143,1.654848,1.892857,1.496169,1.703704,1.717337,False,firing_line,reddy,True
2,game,plan,0.247692,0.890182,0.481358,0.824684,0.644451,0.525238,2.821429,1.964935,4.862069,0.344828,3.827586,1.233693,False,game_plan,reddy,True
3,application,form,0.755922,0.731467,0.862028,0.925423,0.322450,0.786651,4.766667,0.422953,4.862069,0.344828,4.800000,0.476095,False,application_form,reddy,True
4,snail,mail,0.628493,0.973684,0.437938,0.704980,0.653615,0.633459,0.600000,0.800000,4.586207,1.099129,1.310345,1.020596,False,snail_mail,reddy,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282,wedding,day,0.798301,0.864891,0.965938,0.945411,0.798297,0.937918,4.764700,0.562300,4.058800,1.434900,4.941200,0.242500,False,wedding_day,cordeiro100,True
283,white,noise,0.849273,0.941254,0.781347,0.962362,0.949819,0.949227,0.652200,1.112300,4.043500,1.429500,1.173900,1.230400,True,white_noise,cordeiro100,True
284,white,spirit,0.228865,0.481126,0.874926,0.880490,0.949819,0.949227,1.538500,1.240300,2.038500,1.949000,1.307700,1.257600,True,white_spirit,cordeiro100,True
285,winter,solstice,0.970936,0.983268,0.993550,0.999743,0.924877,0.942074,5.000000,0.000000,4.681800,1.086100,4.545500,1.335500,False,winter_solstice,cordeiro100,True


In [20]:
temp_files=glob.glob(f'{args.outputdir}/temporal*')

info_list=[]
for cur_file in temp_files:
    cur_df=pd.read_csv(cur_file,sep='\t')
    cur_shape=cur_df.shape[0]
    info_list.append((cur_file,cur_shape))

In [24]:
google_df=pd.DataFrame(info_list)
google_df

,0,1
0,/mount/studenten/DH_Master/compounding/google/...,287
1,/mount/studenten/DH_Master/compounding/google/...,287
2,/mount/studenten/DH_Master/compounding/google/...,287
3,/mount/studenten/DH_Master/compounding/google/...,287
4,/mount/studenten/DH_Master/compounding/google/...,287
...,...,...
187,/mount/studenten/DH_Master/compounding/google/...,287
188,/mount/studenten/DH_Master/compounding/google/...,287
189,/mount/studenten/DH_Master/compounding/google/...,287
190,/mount/studenten/DH_Master/compounding/google/...,287


In [28]:
temp_files=glob.glob(f'/mount/studenten/DH_Master/compounding/coha/temporal*_UnTagged*')

info_list=[]
for cur_file in temp_files:
    cur_df=pd.read_csv(cur_file,sep='\t')
    cur_shape=cur_df.shape[0]
    info_list.append((cur_file,cur_shape))

In [29]:
coha_df=pd.DataFrame(info_list)
coha_df

,0,1
0,/mount/studenten/DH_Master/compounding/coha/te...,119
1,/mount/studenten/DH_Master/compounding/coha/te...,209
2,/mount/studenten/DH_Master/compounding/coha/te...,97
3,/mount/studenten/DH_Master/compounding/coha/te...,215
4,/mount/studenten/DH_Master/compounding/coha/te...,60
...,...,...
187,/mount/studenten/DH_Master/compounding/coha/te...,182
188,/mount/studenten/DH_Master/compounding/coha/te...,209
189,/mount/studenten/DH_Master/compounding/coha/te...,260
190,/mount/studenten/DH_Master/compounding/coha/te...,260
